# Cohort Selection for Longitudinal ECG Analysis

This notebook identifies patients from the MIMIC-IV and MIMIC-IV ECG database had no heart failure (HF) diagnosis at baseline (at the time of ECG) and separates them into patients who subsequently developed HF and those who remained HF-free during follow-up. The resulting cohort and relevant dates are prepared for subsequent time-to-event analysis of longitudinal ECG biomarkers.

#### To download the MIMIC IV data please visit:
https://physionet.org/content/mimiciv/3.1/


#### To download the MIMIC IV-ECG data please visit: 
https://physionet.org/content/mimic-iv-ecg/1.0/

#### Please cite the original database papers when using this data:

Johnson, A.E.W., Bulgarelli, L., Shen, L. et al. MIMIC-IV, a freely accessible electronic health record dataset. Sci Data 10, 1 (2023). https://doi.org/10.1038/s41597-022-01899-x

Gow, B., Pollard, T., Nathanson, L. A., Johnson, A., Moody, B., Fernandes, C., Greenbaum, N., Waks, J. W., Eslami, P., Carbonati, T., Chaudhari, A., Herbst, E., Moukheiber, D., Berkowitz, S., Mark, R., & Horng, S. (2023). MIMIC-IV-ECG: Diagnostic Electrocardiogram Matched Subset (version 1.0). PhysioNet. RRID:SCR_007345. https://doi.org/10.13026/4nqg-sb35

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
##### Please provide path to the the location of MIMIC-IV data on your device 
path = ''
diagnoses_naming =  pd.read_csv(os.path.join(path,'files/mimiciv/3.1/hosp/d_icd_diagnoses.csv'))
diagnoses  = pd.read_csv(os.path.join(path,'files/mimiciv/3.1/hosp/diagnoses_icd.csv'))

In [ ]:
# HF icd9 and icd10 codes
icd_codes = [
    'I50', 'I500', 'I501', 'I509',
    'I501', 'I502', 'I5020', 'I5021', 'I5022', 'I5023',
    'I503', 'I5030', 'I5031', 'I5032', 'I5033',
    'I504', 'I5040', 'I5041', 'I5042', 'I5043',
    'I508', 'I5081', 'I50810', 'I50811', 'I50812', 'I50813', 
    'I5084', 'I5089', 'I509'
]

diagnoses_naming[diagnoses_naming['icd_code'].isin(icd_codes)]


In [ ]:
### Filter HF ICD codes
df_hf = diagnoses[diagnoses['icd_code'].isin(icd_codes)]
df_hf.head()

In [ ]:
# Read the hospital admissions to connect ICD codes to a specific admission
admissions = pd.read_csv(os.path.join(path,'files/mimiciv/3.1/hosp/admissions.csv'))
#admissions = admissions[admissions['subject_id'].isin(ecgs['subject_id'])]
admissions = admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime']]

In [ ]:
# Connecy ICD code to a specific admission
df_hf = pd.merge(df_hf, admissions[['hadm_id', 'admittime', 'dischtime']], on = 'hadm_id', how = 'inner') 
df_hf['admittime'] = pd.to_datetime(df_hf['admittime'])

In [ ]:
# Find first HF admission to get the first HF diagnosis date
df_hf = df_hf.loc[df_hf.groupby('subject_id')['admittime'].idxmin()]

In [ ]:
# Store the CSV with patient ID and HF diagnosis date
#df_hf.to_csv('HF_codes_MIMIC.csv', index = False)

In [ ]:
# Store last visit date of patients that do not have HF recorded
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['admittime'] = pd.to_datetime(admissions['admittime'])

df_last_max = (
    admissions.loc[
        admissions.groupby('subject_id')['dischtime'].idxmax()
    ]
    .reset_index(drop=True)
)

#df_last_max[['subject_id', 'dischtime']].to_csv('last_visit_date_MIMIC.csv', index = False)